# Show saved averaged templates

Load every `round_XXX/template.npy` written by `learn_averaged_templates_from_gpl(..., template_out_dir=…)` and plot them in a grid.


In [ ]:
%matplotlib inline

from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

TEMPLATE_DIR = Path(
    "/home/yfreund/whale_processing/result/LeiData/dict_learning_templates/n90_r50"
)
MANIFEST = TEMPLATE_DIR / "manifest.json"

# Display grid (rows × cols). Adjust if you change how many rounds were saved.
NROWS, NCOLS = 5, 10

print(f"TEMPLATE_DIR: {TEMPLATE_DIR}")
print(f"exists: {TEMPLATE_DIR.is_dir()}")


## Load templates from disk


In [ ]:
if not MANIFEST.is_file():
    raise FileNotFoundError(f"Missing manifest: {MANIFEST}")

manifest = json.loads(MANIFEST.read_text())
rounds = sorted(manifest["rounds"], key=lambda r: r["round_index"])

templates = []
for entry in rounds:
    npy_path = TEMPLATE_DIR / entry["template_file"]
    data = np.load(npy_path)
    templates.append(
        {
            "round_index": entry["round_index"],
            "template_id": entry["template_id"],
            "data": data,
            "shape": tuple(data.shape),
            "delta_energy": entry.get("delta_energy"),
            "n_matchings": entry.get("n_matchings"),
            "source_segment_id": entry.get("source_segment_id"),
            "path": npy_path,
        }
    )

print(f"Loaded {len(templates)} templates from {TEMPLATE_DIR}")
for t in templates[:5]:
    print(
        f"  round {t['round_index']:03d}: shape={t['shape']}  "
        f"m={t['n_matchings']}  ΔE={t['delta_energy']:.3e}"
    )
if len(templates) > 5:
    print("  …")
    t = templates[-1]
    print(
        f"  round {t['round_index']:03d}: shape={t['shape']}  "
        f"m={t['n_matchings']}  ΔE={t['delta_energy']:.3e}"
    )


## Plot helpers


In [ ]:
def percentile_scale(arr, lo=5.0, hi=99.5):
    arr = np.asarray(arr)
    p_lo, p_hi = np.percentile(arr, [lo, hi])
    return float(p_lo), float(p_hi)


def normalize_and_root(arr, lo, hi):
    arr = np.clip((arr - lo) / (hi - lo + 1e-12), 0, 1)
    return np.cbrt(arr)


## All templates (shared color scale)


In [ ]:
n = len(templates)
if n == 0:
    raise RuntimeError("No templates to plot")
if n > NROWS * NCOLS:
    raise RuntimeError(
        f"Need a larger grid: {n} templates > {NROWS}×{NCOLS}. "
        f"Set NROWS/NCOLS in the setup cell."
    )

all_vals = np.concatenate([t["data"].ravel() for t in templates])
lo, hi = percentile_scale(all_vals, lo=5, hi=99.5)

fig, axes = plt.subplots(NROWS, NCOLS, figsize=(2.2 * NCOLS, 2.4 * NROWS))
axes_flat = np.atleast_1d(axes).ravel()

im = None
for idx, ax in enumerate(axes_flat):
    if idx >= n:
        ax.axis("off")
        continue
    t = templates[idx]
    img = normalize_and_root(t["data"], lo, hi)
    # extent uses bin indices so different widths stay comparable as panels
    f_bins, w_bins = t["data"].shape
    im = ax.imshow(
        img,
        origin="lower",
        aspect="auto",
        cmap="jet",
        extent=[0, w_bins, 0, f_bins],
        vmin=0,
        vmax=1,
    )
    ax.set_title(
        f"r{t['round_index']}  {f_bins}×{w_bins}\n"
        f"m={t['n_matchings']}  ΔE={t['delta_energy']:.2e}",
        fontsize=8,
    )
    ax.tick_params(labelsize=6)
    if idx % NCOLS == 0:
        ax.set_ylabel("freq bin", fontsize=7)
    if idx >= (NROWS - 1) * NCOLS or idx >= n - NCOLS:
        ax.set_xlabel("time bin", fontsize=7)

if im is not None:
    fig.colorbar(
        im,
        ax=axes_flat[:n].tolist(),
        fraction=0.015,
        pad=0.01,
        label="shared [5–99.5%]+root",
    )
fig.suptitle(f"Averaged templates ({n}) — {TEMPLATE_DIR.name}", y=1.01)
fig.tight_layout()
plt.show()


## Energy removed per round


In [ ]:
round_ids = [t["round_index"] for t in templates]
deltas = [t["delta_energy"] for t in templates]
match_counts = [t["n_matchings"] for t in templates]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(round_ids, deltas, color="steelblue")
axes[0].set_xlabel("Outer round")
axes[0].set_ylabel("ΔE")
axes[0].set_title("Energy removed per round")
axes[0].set_xticks(round_ids[:: max(1, len(round_ids) // 10)])
axes[0].grid(True, axis="y", alpha=0.3)

axes[1].plot(round_ids, match_counts, marker="o", markersize=3)
axes[1].set_xlabel("Outer round")
axes[1].set_ylabel("# matchings")
axes[1].set_title("Matchings per round")
axes[1].set_xticks(round_ids[:: max(1, len(round_ids) // 10)])
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()
